# OSNet: повторная проверка GeM и MixStyle

В `suite_v1` G1/S2 фактически повторяли B0 из-за ошибки передачи настроек в конструктор. Здесь проверяются **исправленные G1_gem и S2_mixstyle** против сохранённого B0. Старые результаты не переписываются и не считаются проверкой этих гипотез.

До 10 новых обучений: 2 варианта × 3 seed на primary; победитель против B0 на alternate (до 3 новых); один refit победителя. B0 берётся из завершённого `suite_v1` только после проверки совместимости и SHA256. Это отдельное сравнение, не новый выбор среди всех 22 вариантов.

BBox, метки, строки, исходные splits/validation и MVP остаются неизменными. Интернет и новые веса не нужны. См. [README](README.md).


In [1]:
from pathlib import Path
import gc
import os
import sys

candidates = [Path.cwd(), Path.cwd() / 'Car-classification-MSK', *Path.cwd().parents]
REPO = next((p for p in candidates if (p / 'training/osnet_ablation_suite.py').is_file()), None)
if REPO is None:
    raise RuntimeError('Открой notebook из каталога проекта Car-classification-MSK')
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')

import torch
import pandas as pd
from dataclasses import asdict
from IPython.display import Markdown, display
from training.osnet import GeM, MixStyle
from training.osnet_ablations import experiment_grid, initialize
from training.osnet_ablation_suite import Budget, prepare, run_suite

torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.use_deterministic_algorithms(True, warn_only=True)
print('Repository:', REPO)
print('Torch:', torch.__version__)


Repository: /Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK
Torch: 2.14.0


## 1. Настройки

Запуск: **Restart Kernel → Run All** в прежнем окружении `.venv`. Для повторного использования B0 устройство и версии библиотек должны совпадать с `suite_v1` (MPS). Не меняй старый notebook, `suite_v1` или его manifest.

Если нужен другой backend/бюджет/окружение: задай новый `RUN_NAME` и `REUSE_CONTROL_FROM = None`. Тогда B0 обучится заново в этой серии; это уже до 17 новых обучений. Нельзя обходить проверки старого контроля.


In [2]:
RUN_NAME = 'suite_v2_gem_mixstyle_fix'
REUSE_CONTROL_FROM = 'suite_v1'
DEVICE = 'mps'  # Тот же backend, что у сохранённого B0.
ALLOW_CPU_TRAINING = False
BUDGET = Budget(max_steps=1700, evaluation_interval=200, warmup_steps=100)
SEEDS = (20260915, 20260916, 20260917)
VARIANT_NAMES = ['B0_control', 'G1_gem', 'S2_mixstyle']
FINAL_EVALUATION = True

print('Device:', DEVICE)
print('Budget:', asdict(BUDGET))
print('Control reference:', REUSE_CONTROL_FROM)
display(pd.DataFrame([{'variant': name, 'hypothesis': experiment_grid()[name].description}
                     for name in VARIANT_NAMES]))


Device: mps
Budget: {'max_steps': 1700, 'evaluation_interval': 200, 'warmup_steps': 100}
Control reference: suite_v1


,variant,hypothesis
0,B0_control,"Stock-start, исходный рецепт, фиксированный st..."
1,G1_gem,GeM train/inference от stock
2,S2_mixstyle,MixStyle на честном holdout


## 2. Preflight — без обучения

Проверяем 9556 исходных изображений, CSV, splits, recipe, веса, mask cache, evaluator и runtime. Для B0 разрешено только проверенное изменение конструктора/обвязки; остальные исходники обязаны совпадать. Хеши старых B0-артефактов фиксируются в новом manifest и повторно проверяются при запуске/возобновлении.

Затем выводим **реальные классы модулей**. У G1 должен быть `GeM`, у S2 — `MixStyle`. Проверка конфигурации сама по себе недостаточна.


In [3]:
context = prepare(RUN_NAME, BUDGET, SEEDS, VARIANT_NAMES, device=DEVICE,
                  reuse_control_from=REUSE_CONTROL_FROM)
print('Output:', context['output'])
print('Rows:', len(context['rows']))
print('Outer identities:', {k: len(v) for k, v in context['manifest']['outer'].items()})
print('Source/protocol fingerprint:', context['signature'])

architecture = []
for name, variant in context['variants'].items():
    config = variant.recipe(context['base'], SEEDS[0])
    model = initialize(2, config, variant, 'cpu')
    pool, style = model.backbone.global_pool, model.backbone.mixstyle
    assert isinstance(pool, GeM) == (name == 'G1_gem')
    assert isinstance(style, MixStyle) == (name == 'S2_mixstyle')
    architecture.append({'variant': name, 'pooling': type(pool).__name__,
                         'style': type(style).__name__, 'BNNeck': type(model.bnneck).__name__,
                         'GeM p': float(pool.p.detach()) if isinstance(pool, GeM) else None})
    del model, pool, style
_ = gc.collect()
display(pd.DataFrame(architecture))
print('Preflight passed. Training has not started yet.')


Source integrity (no edits):   0%|          | 0/9556 [00:00<?, ?it/s]

Output: /Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/OSNet-AIN-x1.0/variant_14_controlled_ablations/runs/suite_v2_gem_mixstyle_fix
Rows: 9556
Outer identities: {'train': 925, 'calibration': 307, 'validation': 309}
Source/protocol fingerprint: e80848ab5f96fecbc892cb179b838fe395ef38db044aa98d81dd9867df93ff4a


,variant,pooling,style,BNNeck,GeM p
0,B0_control,AdaptiveAvgPool2d,Identity,BatchNorm1d,NaN
1,G1_gem,GeM,Identity,BatchNorm1d,3.0
2,S2_mixstyle,AdaptiveAvgPool2d,MixStyle,BatchNorm1d,NaN


Preflight passed. Training has not started yet.


## 3. Обучение и итоговая оценка

Оба исправленных варианта проходят все три seed; победитель выбирается только по mean raw mAP@10 primary. Alternate split проверяет устойчивость, но не меняет выбор. Число шагов refit — медиана best step на primary; calibration выбирает пороги до оценки исходной validation.

Сохраняется прежний вывод: вариант/seed, step/1700, inner mAP, best mAP, elapsed и ETA. Контроль печатает `reused ... (no training)`.

При прерывании: Restart Kernel → Run All с теми же настройками. Повторится не более одного незавершённого блока. Не запускай одну серию в двух kernel и не меняй код/данные/окружение во время работы.


In [4]:
if DEVICE == 'cpu' and not ALLOW_CPU_TRAINING:
    raise RuntimeError('CPU-обучение отключено. Выбери MPS/CUDA или явно ALLOW_CPU_TRAINING=True.')
result = run_suite(context, final_evaluation=FINAL_EVALUATION)
print('Finished. MVP promoted:', result['promoted'])


primary/B0_control/20260915: reused from suite_v1 (no training)
primary/G1_gem/20260915: 200/1700 | inner mAP=0.7879, best=0.7879 | elapsed 00:03:03 | ETA 00:22:56
primary/G1_gem/20260915: 285/1700 | inner mAP=0.7849, best=0.7879 | elapsed 00:04:26 | ETA 00:22:02
primary/G1_gem/20260915: 400/1700 | inner mAP=0.7906, best=0.7906 | elapsed 00:06:17 | ETA 00:20:24
primary/G1_gem/20260915: 600/1700 | inner mAP=0.8109, best=0.8109 | elapsed 00:09:15 | ETA 00:16:57
primary/G1_gem/20260915: 800/1700 | inner mAP=0.8153, best=0.8153 | elapsed 00:12:13 | ETA 00:13:45
primary/G1_gem/20260915: 850/1700 | inner mAP=0.8045, best=0.8153 | elapsed 00:13:04 | ETA 00:13:04
primary/G1_gem/20260915: 1000/1700 | inner mAP=0.8119, best=0.8153 | elapsed 00:15:21 | ETA 00:10:44
primary/G1_gem/20260915: 1200/1700 | inner mAP=0.8110, best=0.8153 | elapsed 00:18:20 | ETA 00:07:38
primary/G1_gem/20260915: 1400/1700 | inner mAP=0.8138, best=0.8153 | elapsed 00:21:19 | ETA 00:04:34
primary/G1_gem/20260915: 1600/170

/Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/.venv/lib/python3.11/site-packages/torch/onnx/_internal/torchscript_exporter/symbolic_opset9.py:2855: UserWarning: ONNX export mode is set to TrainingMode.EVAL, but operator 'instance_norm' is set to train=True. Exporting with train=True.
  symbolic_helper.check_training_mode(use_input_stats, "instance_norm")


Finished. MVP promoted: False


## 4. Результаты

Основные сравнения — три paired seed, alternate split, исходная outer validation и bootstrap относительно B0. Masked условия — только диагностика, исходная validation не меняется. Старые T1/M3 можно обсуждать отдельно; этот запуск не переопределяет победителя всей предыдущей серии.

Validation уже использовалась в исследованиях: это development, не независимый финальный тест. Рост метрики одного seed не гарантирует улучшение. Новые checkpoint/ONNX остаются в каталоге серии и автоматически не заменяют MVP.


In [5]:
display(Markdown((context['output'] / 'RESULTS.md').read_text(encoding='utf-8')))
print('Detailed artifacts:', context['output'])


# OSNet variant 14 — результаты

Исходные строки/bbox/validation не изменены. MVP не заменён.
Selection: primary inner raw mAP@10; outer используется только после freeze выбора.

B0 повторно использован из suite_v1 без переобучения; provenance в manifest.json.
Это отдельное сравнение исправленных вариантов с B0, не новый выбор среди всех 22 вариантов.

## Screening (один seed, не доказательство)

| Вариант | inner mAP@10 | best step | samples seen |
|---|---:|---:|---:|
| B0_control | 0.81612 | 1600 | 54400 |
| G1_gem | 0.81833 | 1600 | 54400 |
| S2_mixstyle | 0.81602 | 850 | 54400 |

## Confirmation: три seed

| Вариант | mean | sample std | final steps |
|---|---:|---:|---:|
| B0_control | 0.82093 | 0.00420 | 1600 |
| G1_gem | 0.81669 | 0.00655 | 1600 |
| S2_mixstyle | 0.82481 | 0.00969 | 850 |

Замороженный выбор: **S2_mixstyle**. Альтернативный split не меняет выбор.

## Alternate frame-grouped split

| Вариант | seed | inner mAP@10 |
|---|---:|---:|
| B0_control | 20260915 | 0.81555 |
| B0_control | 20260916 | 0.82261 |
| B0_control | 20260917 | 0.81564 |
| S2_mixstyle | 20260915 | 0.81011 |
| S2_mixstyle | 20260916 | 0.82607 |
| S2_mixstyle | 20260917 | 0.81414 |

## Outer validation — только отчёт

| Вариант / input | raw mAP@10 | reranked mAP@10 | F1 | TNR | 0.7F1+0.3TNR |
|---|---:|---:|---:|---:|---:|
| MVP / frozen reference | 0.79016 | 0.81469 | 0.72861 | 0.79032 | 0.74712 |
| B0_control / original | 0.77769 | 0.79215 | 0.77477 | 0.59677 | 0.72137 |
| B0_control / masked_query | 0.75731 | 0.77608 | 0.76389 | 0.67742 | 0.73795 |
| B0_control / masked_gallery | 0.75470 | 0.77383 | 0.75751 | 0.64516 | 0.72380 |
| B0_control / masked_both | 0.73477 | 0.76054 | 0.74126 | 0.62903 | 0.70759 |
| S2_mixstyle / original | 0.75641 | 0.76999 | 0.75058 | 0.66129 | 0.72380 |
| S2_mixstyle / masked_query | 0.74994 | 0.76163 | 0.72249 | 0.67742 | 0.70897 |
| S2_mixstyle / masked_gallery | 0.74229 | 0.75497 | 0.73709 | 0.64516 | 0.70951 |
| S2_mixstyle / masked_both | 0.73621 | 0.74544 | 0.72813 | 0.64516 | 0.70324 |

Validation уже использовалась в исследованиях: это development, не независимый финальный тест.
MVP reference взят из проверенного baseline_metrics.json, не из нового обучения; hashes зафиксированы.
Mask-условия — дополнительные diagnostics; основная validation остаётся original.
CPU forward timing не заменяет официальный A5000 extract benchmark. Автопродвижения нет.


Detailed artifacts: /Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/OSNet-AIN-x1.0/variant_14_controlled_ablations/runs/suite_v2_gem_mixstyle_fix
